In [1]:
import numpy as np 
import pandas as pd 

In [2]:
df = pd.read_parquet('../data/Divar_cleaned.parquet')
city = pd.read_csv("../data/iran_city_classification.csv")

In [10]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# ۱. ادغام دیتای اصلی با فایل دسته‌بندی شهرها
# اتصال ستون city_slug از فایل اصلی به ستون 'نام شهر' از فایل تکمیلی
merged_df = df.merge(
    city, left_on="city_slug", right_on="نام شهر", how="inner"
)

# ۲. جداسازی متراژ بنا (building_size) برای دو گروه کلان‌شهر و شهر کوچک
# (حذف مقادیر مفقود و متراژهای صفر یا نامعتبر)
metro_sizes = merged_df[
    (merged_df["دسته‌بندی"] != "شهر کوچک") & (merged_df["building_size"] > 0)
]["building_size"].dropna()

small_town_sizes = merged_df[
    (merged_df["دسته‌بندی"] == "شهر کوچک") & (merged_df["building_size"] > 0)
]["building_size"].dropna()

# ۳. محاسبه میانگین‌ها و آمار توصیفی
mean_metro = metro_sizes.mean()
mean_small = small_town_sizes.mean()

print(f"تعداد آگهی‌های کلان‌شهرها: {len(metro_sizes):,}")
print(f"تعداد آگهی‌های شهرهای کوچک/روستاها: {len(small_town_sizes):,}")
print("-" * 55)
print(f"میانگین متراژ بنا در کلان‌شهرها: {mean_metro:.2f} متر مربع")
print(f"میانگین متراژ بنا در شهرهای کوچک: {mean_small:.2f} متر مربع")

# ۴. اجرای آزمون t ولچ (Welch's t-test)
t_stat, p_value_two_tailed = stats.ttest_ind(
    metro_sizes, small_town_sizes, equal_var=False
)

# تبدیل p-value به یک‌دامنه (چون ادعا کم‌تر بودن میانگین است)
p_value_one_tailed = (
    p_value_two_tailed / 2 if t_stat < 0 else 1 - (p_value_two_tailed / 2)
)

alpha = 0.05  # سطح معنی‌داری ۵ درصد

print("-" * 55)
print(f"آمار آزمون (t-statistic): {t_stat:.4f}")
print(f"مقدار p-value یک‌دامنه: {p_value_one_tailed:.4e}")

# ۵. نتیجه‌گیری نهایی
print("-" * 55)
if p_value_one_tailed < alpha and t_stat < 0:
    print(
        "✅ فرضیه صفر رد می‌شود: مجموعه داده از این ادعا پشتیبانی می‌کند\n"
        "(میانگین مساحت خانه‌ها در کلان‌شهرها به طور معنی‌داری کمتر از شهرهای کوچک است)."
    )
else:
    print(
        "❌ فرضیه صفر رد نمی‌شود: شواهد کافی برای تایید ادعا در مجموعه داده وجود ندارد."
    )

تعداد آگهی‌های کلان‌شهرها: 446,877
تعداد آگهی‌های شهرهای کوچک/روستاها: 469,222
-------------------------------------------------------
میانگین متراژ بنا در کلان‌شهرها: 160.42 متر مربع
میانگین متراژ بنا در شهرهای کوچک: 212.49 متر مربع
-------------------------------------------------------
آمار آزمون (t-statistic): -39.9018
مقدار p-value یک‌دامنه: 0.0000e+00
-------------------------------------------------------
✅ فرضیه صفر رد می‌شود: مجموعه داده از این ادعا پشتیبانی می‌کند
(میانگین مساحت خانه‌ها در کلان‌شهرها به طور معنی‌داری کمتر از شهرهای کوچک است).


<div dir="rtl">

### تحلیل و پیاده‌سازی آزمون فرض (ارزیابی میانگین مساحت املاک)

* **ادغام مجموعه‌داده‌ها:** تطبیق ستون `city_slug` از دیتای اصلی دیوار با ستون `نام شهر` از فایل دسته‌بندی `iran_city_classification.csv` جهت تفکیک دقیق آگهی‌های کلان‌شهر از شهرهای کوچک.
* **صورت‌بندی فرضیه‌ها:**
  * **فرضیه صفر ($H_0$):** میانگین مساحت بنا در کلان‌شهرها بیشتر یا برابر با شهرهای کوچک است ($\mu_{metro} \ge \mu_{small}$).
  * **فرضیه جایگزین ($H_1$):** میانگین مساحت بنا در کلان‌شهرها کمتر از شهرهای کوچک است ($\mu_{metro} < \mu_{small}$).
* **آزمون آماری انتخاب‌شده:** آزمون **t-دو نمونه‌ای ولچ (Welch's t-test)** به علت ناهمگونی واریانس‌ها و تفاوت شدید در حجم نمونه دو گروه با سطح معنی‌داری $\alpha = 0.05$.

---

#### 💡 نتیجه‌گیری آزمون
با بررسی مقادیر آماری به‌دست‌آمده، در صورتی که $p\text{-value} < 0.05$ و آمار آزمون $t < 0$ باشد، فرضیه صفر رد شده و ادعای کمتر بودن میانگین متراژ در کلان‌شهرها با اطمینان ۹۵٪ از سوی داده‌ها **تایید می‌گردد**.

</div>